In [1]:
import pandas as pd
import numpy as np
import os

# Project paths
project_path = "/Users/simrannimje/Desktop/Collections_Analytics_Assignment"

cleaned_path = os.path.join(
    project_path, "Data", "Cleaned"
)

output_path = os.path.join(
    project_path, "Outputs"
)

os.makedirs(output_path, exist_ok=True)

# Load cleaned datasets
accounts = pd.read_csv(
    os.path.join(cleaned_path, "accounts_cleaned.csv")
)

payments = pd.read_csv(
    os.path.join(cleaned_path, "payments_cleaned.csv")
)

# Convert dates
accounts["opened_at"] = pd.to_datetime(
    accounts["opened_at"], errors="coerce"
)

payments["event_at"] = pd.to_datetime(
    payments["event_at"], errors="coerce"
)

# Display dataset information
print("ACCOUNTS")
print("Shape:", accounts.shape)
print("Unique account IDs:", accounts["account_id"].nunique())

print("\nPAYMENTS")
print("Shape:", payments.shape)
print("Unique payment IDs:", payments["payment_id"].nunique())

print("\nPayment statuses:")
display(payments["payment_status"].value_counts())

ACCOUNTS
Shape: (30000, 13)
Unique account IDs: 30000

PAYMENTS
Shape: (25000, 9)
Unique payment IDs: 25000

Payment statuses:


payment_status
SUCCESS     17534
FAILED       3677
PENDING      2535
REVERSED     1254
Name: count, dtype: int64

In [2]:
#validating the account and payment realationship
# Check account-payment relationship

# Find payment records without a matching account
payment_account_check = payments.merge(
    accounts[["account_id"]],
    on="account_id",
    how="left",
    indicator=True
)

print("Total payment records:", len(payment_account_check))

print("\nPayment-to-account mapping:")
display(payment_account_check["_merge"].value_counts())

print(
    "\nPayments without matching accounts:",
    (payment_account_check["_merge"] == "left_only").sum()
)

# Check accounts receiving multiple payments
account_payment_frequency = payments.groupby(
    "account_id"
).agg(
    payment_count=("payment_id", "count"),
    total_amount=("amount", "sum")
).reset_index()

print(
    "\nAccounts with multiple payment transactions:",
    (account_payment_frequency["payment_count"] > 1).sum()
)

print(
    "Unique accounts with payments:",
    payments["account_id"].nunique()
)

print(
    "Maximum payments for one account:",
    account_payment_frequency["payment_count"].max()
)

Total payment records: 25000

Payment-to-account mapping:


_merge
both          25000
left_only         0
right_only        0
Name: count, dtype: int64


Payments without matching accounts: 0

Accounts with multiple payment transactions: 6114
Unique accounts with payments: 16934
Maximum payments for one account: 7


In [3]:
# Check payment amounts across different statuses

payment_status_summary = payments.groupby("payment_status").agg(
    transaction_count=("payment_id", "count"),
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean"),
    unique_accounts=("account_id", "nunique")
).reset_index()

display(payment_status_summary.round(2))

# Check missing and invalid payment amounts

print("Missing payment amounts:", payments["amount"].isna().sum())
print("Zero or negative payment amounts:", (payments["amount"] <= 0).sum())

# Check payment date range

print("First payment date:", payments["event_at"].min())
print("Last payment date:", payments["event_at"].max())

,payment_status,transaction_count,total_amount,average_amount,unique_accounts
0,FAILED,3677,2.784186e+08,75718.95,3443
1,PENDING,2535,1.902232e+08,75038.73,2430
2,REVERSED,1254,9.466657e+07,75491.68,1230
3,SUCCESS,17534,1.315584e+09,75030.45,13284


Missing payment amounts: 0
Zero or negative payment amounts: 0
First payment date: 2026-01-01 00:14:40
Last payment date: 2026-08-08 23:50:23


In [4]:
# Retain only successful payments for recovery analysis

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
].copy()

# Create monthly payment period

successful_payments["payment_month"] = (
    successful_payments["event_at"].dt.to_period("M")
)

# Monthly recovery summary

monthly_recovery = successful_payments.groupby(
    "payment_month"
).agg(
    successful_transactions=("payment_id", "count"),
    recovered_amount=("amount", "sum"),
    unique_paying_accounts=("account_id", "nunique"),
    average_payment=("amount", "mean")
).reset_index()

monthly_recovery["recovered_amount_cr"] = (
    monthly_recovery["recovered_amount"] / 10000000
)

display(monthly_recovery.round(2))

/var/folders/v_/4fg80cgj7sj9t07y2dc08t680000gn/T/ipykernel_32399/483348231.py:28: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(monthly_recovery.round(2))


,payment_month,successful_transactions,recovered_amount,unique_paying_accounts,average_payment,recovered_amount_cr
0,2026-01,2464,1.872291e+08,2374,75985.85,18.72
1,2026-02,2268,1.701425e+08,2173,75018.72,17.01
2,2026-03,2524,1.889124e+08,2419,74846.42,18.89
3,2026-04,2406,1.751380e+08,2304,72792.20,17.51
4,2026-05,2449,1.842503e+08,2344,75234.90,18.43
5,2026-06,2366,1.755597e+08,2286,74201.07,17.56
6,2026-07,2441,1.872423e+08,2335,76707.20,18.72
7,2026-08,616,4.710970e+07,608,76476.78,4.71


In [5]:
# Calculate month-on-month recovery growth

monthly_recovery["mom_growth_pct"] = (
    monthly_recovery["recovered_amount"].pct_change() * 100
)

# Calculate January-to-July overall change

jan_recovery = monthly_recovery.loc[
    monthly_recovery["payment_month"] == "2026-01",
    "recovered_amount"
].iloc[0]

jul_recovery = monthly_recovery.loc[
    monthly_recovery["payment_month"] == "2026-07",
    "recovered_amount"
].iloc[0]

overall_change = ((jul_recovery - jan_recovery) / jan_recovery) * 100

# Average monthly growth from February to July

average_growth = monthly_recovery.loc[
    monthly_recovery["payment_month"].between("2026-02", "2026-07"),
    "mom_growth_pct"
].mean()

print("January to July overall recovery change:", round(overall_change, 2), "%")
print("Average monthly growth (February to July):", round(average_growth, 2), "%")

display(
    monthly_recovery[
        ["payment_month", "recovered_amount_cr", "mom_growth_pct"]
    ].round(2)
)

January to July overall recovery change: 0.01 %
Average monthly growth (February to July): 0.29 %


/var/folders/v_/4fg80cgj7sj9t07y2dc08t680000gn/T/ipykernel_32399/2253382029.py:34: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,payment_month,recovered_amount_cr,mom_growth_pct
0,2026-01,18.72,NaN
1,2026-02,17.01,-9.13
2,2026-03,18.89,11.03
3,2026-04,17.51,-7.29
4,2026-05,18.43,5.20
5,2026-06,17.56,-4.72
6,2026-07,18.72,6.65
7,2026-08,4.71,-74.84


In [6]:
# Aggregate successful payments at account level

account_recovery = successful_payments.groupby("account_id").agg(
    successful_payment_count=("payment_id", "count"),
    total_recovered=("amount", "sum"),
    first_payment_date=("event_at", "min"),
    last_payment_date=("event_at", "max")
).reset_index()

print("Accounts with successful payments:", account_recovery["account_id"].nunique())
print("Total successful payment amount:", round(account_recovery["total_recovered"].sum(), 2))

# Join account information with successful recovery

golden_dataset = accounts.merge(
    account_recovery,
    on="account_id",
    how="left"
)

# Accounts without successful payments receive zero recovery

golden_dataset["successful_payment_count"] = (
    golden_dataset["successful_payment_count"].fillna(0).astype(int)
)

golden_dataset["total_recovered"] = (
    golden_dataset["total_recovered"].fillna(0)
)

print("\nGolden dataset shape:", golden_dataset.shape)
print("Unique accounts:", golden_dataset["account_id"].nunique())
print("Accounts with recovery:", (golden_dataset["total_recovered"] > 0).sum())
print("Accounts without recovery:", (golden_dataset["total_recovered"] == 0).sum())

display(golden_dataset.head())

Accounts with successful payments: 13284
Total successful payment amount: 1315583964.64

Golden dataset shape: (30000, 17)
Unique accounts: 30000
Accounts with recovery: 13284
Accounts without recovery: 16716


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,opened_at,timezone,schema_version,outstanding_above_principal_flag,borrower_id_missing_flag,successful_payment_count,total_recovered,first_payment_date,last_payment_date
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,2025-11-11 04:37:00,Asia/Dubai,v1,True,False,1,114222.84,2026-04-09 17:09:27,2026-04-09 17:09:27
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,2025-11-13 15:59:44,UTC,v3,True,False,0,0.00,NaT,NaT
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2025-09-12 04:59:20,Asia/Dubai,v2,False,False,2,118181.69,2026-02-11 22:46:19,2026-06-22 00:35:27
3,ACC0000004,BRW0001993,PERSONAL,76435.07,508427.73,30,LOW,CLOSED,2025-05-25 19:29:38,UTC,v2,True,False,1,120763.00,2026-06-24 20:53:31,2026-06-24 20:53:31
4,ACC0000005,BRW0009976,CONSUMER,646797.08,563858.01,180,LOW,WRITEOFF,2025-09-07 14:59:13,Asia/Dubai,v2,False,False,0,0.00,NaT,NaT


In [7]:
# Recovery performance by loan type

loan_recovery = golden_dataset.groupby("loan_type").agg(
    total_accounts=("account_id", "nunique"),
    accounts_with_recovery=("total_recovered", lambda x: (x > 0).sum()),
    total_recovered=("total_recovered", "sum"),
    average_recovery=("total_recovered", "mean"),
    total_outstanding=("outstanding_amount", "sum")
).reset_index()

loan_recovery["recovery_coverage_pct"] = (
    loan_recovery["accounts_with_recovery"] /
    loan_recovery["total_accounts"] * 100
)

loan_recovery["recovery_to_outstanding_pct"] = (
    loan_recovery["total_recovered"] /
    loan_recovery["total_outstanding"] * 100
)

display(loan_recovery.round(2))

,loan_type,total_accounts,accounts_with_recovery,total_recovered,average_recovery,total_outstanding,recovery_coverage_pct,recovery_to_outstanding_pct
0,AUTO,6079,2700,2.679332e+08,44075.21,2.135301e+09,44.42,12.55
1,BNPL,5928,2580,2.554400e+08,43090.42,2.058915e+09,43.52,12.41
2,CONSUMER,5930,2685,2.669767e+08,45021.36,2.073113e+09,45.28,12.88
3,CREDIT_CARD,6080,2690,2.686432e+08,44184.74,2.126677e+09,44.24,12.63
4,PERSONAL,5983,2629,2.565909e+08,42886.66,2.095030e+09,43.94,12.25


In [8]:
# Recovery performance by risk segment

risk_recovery = golden_dataset.groupby("risk_segment").agg(
    total_accounts=("account_id", "nunique"),
    accounts_with_recovery=("total_recovered", lambda x: (x > 0).sum()),
    total_recovered=("total_recovered", "sum"),
    total_outstanding=("outstanding_amount", "sum")
).reset_index()

risk_recovery["recovery_coverage_pct"] = (
    risk_recovery["accounts_with_recovery"] /
    risk_recovery["total_accounts"] * 100
)

risk_recovery["recovery_to_outstanding_pct"] = (
    risk_recovery["total_recovered"] /
    risk_recovery["total_outstanding"] * 100
)

display(risk_recovery.round(2))

,risk_segment,total_accounts,accounts_with_recovery,total_recovered,total_outstanding,recovery_coverage_pct,recovery_to_outstanding_pct
0,HIGH,7552,3300,3.315982e+08,2.646183e+09,43.70,12.53
1,LOW,7513,3381,3.308578e+08,2.633311e+09,45.00,12.56
2,MEDIUM,7533,3335,3.305101e+08,2.628179e+09,44.27,12.58
3,NPA,7402,3268,3.226178e+08,2.581362e+09,44.15,12.50


In [9]:
#recovery by dpd bucket
# Create delinquency buckets

dpd_bins = [-1, 0, 30, 60, 90, 120, 180]
dpd_labels = [
    "0 DPD",
    "1-30 DPD",
    "31-60 DPD",
    "61-90 DPD",
    "91-120 DPD",
    "121-180 DPD"
]

golden_dataset["dpd_bucket"] = pd.cut(
    golden_dataset["dpd"],
    bins=dpd_bins,
    labels=dpd_labels
)

# Recovery performance by DPD bucket

dpd_recovery = golden_dataset.groupby(
    "dpd_bucket",
    observed=False
).agg(
    total_accounts=("account_id", "nunique"),
    accounts_with_recovery=("total_recovered", lambda x: (x > 0).sum()),
    total_recovered=("total_recovered", "sum"),
    total_outstanding=("outstanding_amount", "sum")
).reset_index()

dpd_recovery["recovery_coverage_pct"] = (
    dpd_recovery["accounts_with_recovery"] /
    dpd_recovery["total_accounts"] * 100
)

dpd_recovery["recovery_to_outstanding_pct"] = (
    dpd_recovery["total_recovered"] /
    dpd_recovery["total_outstanding"] * 100
)

display(dpd_recovery.round(2))

,dpd_bucket,total_accounts,accounts_with_recovery,total_recovered,total_outstanding,recovery_coverage_pct,recovery_to_outstanding_pct
0,0 DPD,2685,1220,1.174619e+08,9.482186e+08,45.44,12.39
1,1-30 DPD,10880,4719,4.715398e+08,3.834101e+09,43.37,12.30
2,31-60 DPD,5514,2524,2.527771e+08,1.914490e+09,45.77,13.20
3,61-90 DPD,5468,2436,2.387357e+08,1.895868e+09,44.55,12.59
4,91-120 DPD,2759,1225,1.234044e+08,9.759497e+08,44.40,12.64
5,121-180 DPD,2694,1160,1.116651e+08,9.204078e+08,43.06,12.13


In [10]:
# Recovery performance by account status

status_recovery = golden_dataset.groupby("status").agg(
    total_accounts=("account_id", "nunique"),
    accounts_with_recovery=("total_recovered", lambda x: (x > 0).sum()),
    total_recovered=("total_recovered", "sum"),
    total_outstanding=("outstanding_amount", "sum")
).reset_index()

status_recovery["recovery_coverage_pct"] = (
    status_recovery["accounts_with_recovery"] /
    status_recovery["total_accounts"] * 100
)

status_recovery["recovery_to_outstanding_pct"] = (
    status_recovery["total_recovered"] /
    status_recovery["total_outstanding"] * 100
)

display(status_recovery.round(2))

,status,total_accounts,accounts_with_recovery,total_recovered,total_outstanding,recovery_coverage_pct,recovery_to_outstanding_pct
0,ACTIVE,7539,3315,3.329433e+08,2.637397e+09,43.97,12.62
1,CLOSED,7496,3325,3.256834e+08,2.636143e+09,44.36,12.35
2,PAID,7486,3297,3.287247e+08,2.617540e+09,44.04,12.56
3,WRITEOFF,7479,3347,3.282327e+08,2.597956e+09,44.75,12.63


In [11]:
# Save account-level golden dataset

golden_file = os.path.join(
    output_path,
    "golden_account_recovery.csv"
)

golden_dataset.to_csv(golden_file, index=False)

print("Golden dataset saved successfully!")
print("File location:", golden_file)
print("Dataset shape:", golden_dataset.shape)
print("Unique accounts:", golden_dataset["account_id"].nunique())

Golden dataset saved successfully!
File location: /Users/simrannimje/Desktop/Collections_Analytics_Assignment/Outputs/golden_account_recovery.csv
Dataset shape: (30000, 18)
Unique accounts: 30000


In [12]:
# Load cleaned call datasets

calls = pd.read_csv(
    os.path.join(cleaned_path, "calls_cleaned.csv")
)

call_dispositions = pd.read_csv(
    os.path.join(cleaned_path, "call_dispositions_cleaned.csv")
)

# Convert event timestamps

calls["event_at"] = pd.to_datetime(
    calls["event_at"], errors="coerce"
)

call_dispositions["event_at"] = pd.to_datetime(
    call_dispositions["event_at"], errors="coerce"
)

# Inspect datasets

print("CALLS")
print("Shape:", calls.shape)
print("Columns:", calls.columns.tolist())

print("\nCALL DISPOSITIONS")
print("Shape:", call_dispositions.shape)
print("Columns:", call_dispositions.columns.tolist())

print("\nCall disposition codes:")
display(call_dispositions["disposition_code"].value_counts())

CALLS
Shape: (90079, 14)
Columns: ['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone', 'agent_id_missing', 'zero_duration', 'repeated_call_id']

CALL DISPOSITIONS
Shape: (35000, 10)
Columns: ['disposition_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'disposition_code', 'disposition_version', 'disposition_code_original', 'legacy_code_flag']

Call disposition codes:


disposition_code
NO_CONTACT        4010
WRONG_NUMBER      3946
PROMISE_TO_PAY    3926
PTP               3904
PTP_BROKEN        3900
CALLBACK          3872
DISPUTE           3826
PAID              3818
REFUSED           3798
Name: count, dtype: int64

In [13]:
# Validate disposition-to-call relationship

disposition_call_check = call_dispositions.merge(
    calls[["call_id", "account_id"]],
    on="call_id",
    how="left",
    suffixes=("_disposition", "_call"),
    indicator=True
)

print("Total disposition records:", len(call_dispositions))

print("\nCall ID matching:")
display(disposition_call_check["_merge"].value_counts())

# Check account ID consistency

account_mismatch = disposition_call_check[
    (disposition_call_check["_merge"] == "both") &
    (
        disposition_call_check["account_id_disposition"] !=
        disposition_call_check["account_id_call"]
    )
]

print("\nAccount ID mismatches:", len(account_mismatch))

# Check multiple dispositions for the same call

dispositions_per_call = call_dispositions.groupby(
    "call_id"
).size()

print(
    "\nCalls with multiple disposition records:",
    (dispositions_per_call > 1).sum()
)

print(
    "Maximum dispositions for one call:",
    dispositions_per_call.max()
)

Total disposition records: 35000

Call ID matching:


_merge
both          35027
left_only         0
right_only        0
Name: count, dtype: int64


Account ID mismatches: 35027

Calls with multiple disposition records: 5282
Maximum dispositions for one call: 5


In [14]:
# Check uniqueness of call IDs in both datasets

print("CALLS")
print("Total rows:", len(calls))
print("Unique call IDs:", calls["call_id"].nunique())
print("Duplicate call ID rows:", calls["call_id"].duplicated().sum())

print("\nCALL DISPOSITIONS")
print("Total rows:", len(call_dispositions))
print("Unique call IDs:", call_dispositions["call_id"].nunique())
print("Missing call IDs:", call_dispositions["call_id"].isna().sum())

# Inspect repeated call IDs in calls

duplicate_calls = calls[
    calls["call_id"].duplicated(keep=False)
].sort_values("call_id")

print("\nSample repeated call IDs:")
display(
    duplicate_calls[
        ["call_id", "account_id", "event_at", "agent_id"]
    ].head(15)
)

# Inspect sample account mismatches

print("\nSample disposition records:")
display(
    disposition_call_check[
        [
            "call_id",
            "account_id_disposition",
            "account_id_call"
        ]
    ].head(10)
)

CALLS
Total rows: 90079
Unique call IDs: 90000
Duplicate call ID rows: 79

CALL DISPOSITIONS
Total rows: 35000
Unique call IDs: 28971
Missing call IDs: 0

Sample repeated call IDs:


,call_id,account_id,event_at,agent_id
225,CALL0000226,ACC0010288,2026-01-09 11:36:06,AGT0000584
90028,CALL0000226,ACC0010288,2026-01-12 11:36:06,AGT0000584
1605,CALL0001606,ACC0028730,2026-02-28 06:39:17,NaN
90044,CALL0001606,ACC0028730,2026-02-28 06:39:17,AGT0000504
1810,CALL0001811,ACC0020868,2026-05-14 02:18:37,NaN
90054,CALL0001811,ACC0020868,2026-05-14 02:18:37,AGT0000531
3081,CALL0003082,ACC0006316,2026-04-18 14:14:49,NaN
90007,CALL0003082,ACC0006316,2026-04-18 14:14:49,AGT0000120
4228,CALL0004229,ACC0014927,2026-05-16 07:06:49,NaN
90071,CALL0004229,ACC0014927,2026-05-16 07:06:49,AGT0000433



Sample disposition records:


,call_id,account_id_disposition,account_id_call
0,CALL0032272,ACC0029208,ACC0008760
1,CALL0053880,ACC0009584,ACC0000867
2,CALL0048622,ACC0003386,ACC0016903
3,CALL0072651,ACC0027899,ACC0012305
4,CALL0036866,ACC0024345,ACC0017005
5,CALL0078106,ACC0000656,ACC0022369
6,CALL0078570,ACC0015494,ACC0026012
7,CALL0042609,ACC0009000,ACC0026787
8,CALL0009101,ACC0021766,ACC0008500
9,CALL0084384,ACC0004669,ACC0010463


In [15]:
# Create unique call-account combinations
# This avoids multiplying rows because of repeated call IDs

call_account_pairs = calls[
    ["call_id", "account_id"]
].drop_duplicates()

print("Unique call-account combinations:", len(call_account_pairs))
print("Unique call IDs:", call_account_pairs["call_id"].nunique())

# Check disposition-to-call account mapping

mapping_check = call_dispositions.merge(
    call_account_pairs,
    on="call_id",
    how="left",
    suffixes=("_disposition", "_call"),
    indicator=True
)

# Identify matched and mismatched account IDs

mapping_check["account_match"] = (
    mapping_check["account_id_disposition"] ==
    mapping_check["account_id_call"]
)

print("\nCall ID matching:")
display(mapping_check["_merge"].value_counts())

print("\nAccount ID consistency:")
display(mapping_check["account_match"].value_counts())

print(
    "\nAccount mismatch percentage:",
    round((~mapping_check["account_match"]).mean() * 100, 2),
    "%"
)

# Inspect repeated call IDs with multiple accounts

accounts_per_call = call_account_pairs.groupby(
    "call_id"
)["account_id"].nunique()

print(
    "\nCall IDs associated with multiple accounts:",
    (accounts_per_call > 1).sum()
)

Unique call-account combinations: 90000
Unique call IDs: 90000

Call ID matching:


_merge
both          35000
left_only         0
right_only        0
Name: count, dtype: int64


Account ID consistency:


account_match
False    35000
Name: count, dtype: int64


Account mismatch percentage: 100.0 %

Call IDs associated with multiple accounts: 0


In [16]:
# Overall call performance

call_summary = calls.groupby("call_status").agg(
    total_calls=("call_id", "count"),
    unique_accounts=("account_id", "nunique"),
    average_duration=("duration_sec", "mean")
).reset_index()

call_summary["call_share_pct"] = (
    call_summary["total_calls"] / len(calls) * 100
)

display(call_summary.round(2))

print("\nTotal calls:", len(calls))
print("Unique accounts contacted:", calls["account_id"].nunique())
print("Average call duration:", round(calls["duration_sec"].mean(), 2), "seconds")

,call_status,total_calls,unique_accounts,average_duration,call_share_pct
0,ANSWERED,17896,13535,452.95,19.87
1,BUSY,18078,13549,450.69,20.07
2,FAILED,18014,13505,450.92,20.00
3,NO_ANSWER,18092,13568,448.54,20.08
4,VOICEMAIL,17999,13423,452.09,19.98



Total calls: 90079
Unique accounts contacted: 28408
Average call duration: 451.03 seconds


In [17]:
# Analyse call disposition outcomes independently

disposition_summary = call_dispositions.groupby(
    "disposition_code"
).agg(
    total_dispositions=("disposition_id", "count"),
    unique_accounts=("account_id", "nunique"),
    unique_calls=("call_id", "nunique")
).reset_index()

disposition_summary["disposition_share_pct"] = (
    disposition_summary["total_dispositions"] /
    len(call_dispositions) * 100
)

display(disposition_summary.round(2))

print("Total disposition records:", len(call_dispositions))
print("Unique accounts in dispositions:", call_dispositions["account_id"].nunique())
print("Unique calls in dispositions:", call_dispositions["call_id"].nunique())

,disposition_code,total_dispositions,unique_accounts,unique_calls,disposition_share_pct
0,CALLBACK,3872,3590,3765,11.06
1,DISPUTE,3826,3610,3734,10.93
2,NO_CONTACT,4010,3750,3926,11.46
3,PAID,3818,3597,3725,10.91
4,PROMISE_TO_PAY,3926,3670,3843,11.22
5,PTP,3904,3687,3807,11.15
6,PTP_BROKEN,3900,3655,3820,11.14
7,REFUSED,3798,3564,3700,10.85
8,WRONG_NUMBER,3946,3695,3851,11.27


Total disposition records: 35000
Unique accounts in dispositions: 20603
Unique calls in dispositions: 28971


In [18]:
# Load cleaned agent datasets

agents = pd.read_csv(
    os.path.join(cleaned_path, "agents_cleaned.csv")
)

agent_sessions = pd.read_csv(
    os.path.join(cleaned_path, "agent_sessions_cleaned.csv")
)

# Convert timestamps

agents["joined_at"] = pd.to_datetime(
    agents["joined_at"], errors="coerce"
)

agents["updated_at"] = pd.to_datetime(
    agents["updated_at"], errors="coerce"
)

agent_sessions["login_at"] = pd.to_datetime(
    agent_sessions["login_at"], errors="coerce"
)

agent_sessions["logout_at"] = pd.to_datetime(
    agent_sessions["logout_at"], errors="coerce"
)

# Inspect datasets

print("AGENTS")
print("Shape:", agents.shape)
print("Unique agent IDs:", agents["agent_id"].nunique())

print("\nAGENT SESSIONS")
print("Shape:", agent_sessions.shape)
print("Unique agent IDs:", agent_sessions["agent_id"].nunique())

print("\nSession duration summary:")
display(agent_sessions["session_duration_hours"].describe().round(2))

AGENTS
Shape: (30000, 9)
Unique agent IDs: 1000

AGENT SESSIONS
Shape: (15000, 9)
Unique agent IDs: 1000

Session duration summary:


count    15000.00
mean         5.26
std          2.74
min          0.50
25%          2.88
50%          5.24
75%          7.63
max         10.00
Name: session_duration_hours, dtype: float64

In [19]:
# Check agent identity consistency

agent_identity_check = agents.groupby("agent_id").agg(
    record_count=("agent_id", "size"),
    unique_employee_codes=("employee_code", "nunique"),
    unique_names=("agent_name", "nunique"),
    unique_vendors=("vendor_id", "nunique"),
    unique_teams=("team", "nunique"),
    unique_statuses=("status", "nunique")
).reset_index()

print("Total agent IDs:", len(agent_identity_check))

print(
    "\nAgents with multiple employee codes:",
    (agent_identity_check["unique_employee_codes"] > 1).sum()
)

print(
    "Agents with multiple names:",
    (agent_identity_check["unique_names"] > 1).sum()
)

print(
    "Agents with multiple vendors:",
    (agent_identity_check["unique_vendors"] > 1).sum()
)

print(
    "Agents with multiple teams:",
    (agent_identity_check["unique_teams"] > 1).sum()
)

print(
    "Agents with multiple employment statuses:",
    (agent_identity_check["unique_statuses"] > 1).sum()
)

print("\nSample inconsistent agent identities:")

display(
    agent_identity_check[
        (agent_identity_check["unique_employee_codes"] > 1) |
        (agent_identity_check["unique_names"] > 1)
    ].head(10)
)

Total agent IDs: 1000

Agents with multiple employee codes: 1000
Agents with multiple names: 1000
Agents with multiple vendors: 1000
Agents with multiple teams: 1000
Agents with multiple employment statuses: 1000

Sample inconsistent agent identities:


,agent_id,record_count,unique_employee_codes,unique_names,unique_vendors,unique_teams,unique_statuses
0,AGT0000001,23,23,10,10,5,3
1,AGT0000002,23,23,10,12,5,3
2,AGT0000003,28,28,10,12,5,3
3,AGT0000004,28,28,10,12,5,3
4,AGT0000005,29,29,10,15,5,3
5,AGT0000006,37,37,10,15,5,3
6,AGT0000007,30,29,10,15,5,3
7,AGT0000008,29,29,10,12,5,3
8,AGT0000009,25,24,10,12,5,3
9,AGT0000010,26,26,10,15,5,3


In [20]:
# Aggregate session activity

session_summary = agent_sessions.groupby("agent_id").agg(
    total_sessions=("agent_id", "count"),
    total_session_hours=("session_duration_hours", "sum"),
    average_session_hours=("session_duration_hours", "mean"),
    minimum_session_hours=("session_duration_hours", "min"),
    maximum_session_hours=("session_duration_hours", "max")
).reset_index()

print("Total agents with sessions:", session_summary["agent_id"].nunique())
print("Total sessions:", session_summary["total_sessions"].sum())
print(
    "Total recorded session hours:",
    round(session_summary["total_session_hours"].sum(), 2)
)

print("\nSession activity summary:")
display(session_summary.describe().round(2))

print("\nSample session activity:")
display(session_summary.head(10))

Total agents with sessions: 1000
Total sessions: 15000
Total recorded session hours: 78870.89

Session activity summary:


,total_sessions,total_session_hours,average_session_hours,minimum_session_hours,maximum_session_hours
count,1000.00,1000.00,1000.00,1000.00,1000.00
mean,15.00,78.87,5.26,1.15,9.39
std,3.68,22.16,0.77,0.62,0.62
min,6.00,12.63,2.10,0.50,4.59
25%,12.00,63.33,4.73,0.69,9.15
50%,15.00,77.86,5.28,0.97,9.58
75%,18.00,93.71,5.75,1.40,9.83
max,28.00,160.20,7.35,4.20,10.00



Sample session activity:


,agent_id,total_sessions,total_session_hours,average_session_hours,minimum_session_hours,maximum_session_hours
0,AGT0000001,18,87.039722,4.835540,0.800556,8.740000
1,AGT0000002,16,65.223056,4.076441,0.820000,9.166944
2,AGT0000003,13,68.858889,5.296838,0.923333,9.131667
3,AGT0000004,11,61.593611,5.599419,1.451111,8.002778
4,AGT0000005,14,90.296667,6.449762,1.291111,9.837778
5,AGT0000006,9,43.226389,4.802932,1.093889,8.686667
6,AGT0000007,18,104.664722,5.814707,0.538056,9.750556
7,AGT0000008,9,51.517500,5.724167,2.303056,9.504722
8,AGT0000009,13,81.575556,6.275043,1.122500,9.578611
9,AGT0000010,18,106.650556,5.925031,1.374167,9.808333


In [22]:
# Load cleaned PTP dataset

ptp = pd.read_csv(
    os.path.join(cleaned_path, "ptp_cleaned.csv")
)

ptp["event_at"] = pd.to_datetime(ptp["event_at"])
ptp["promised_date"] = pd.to_datetime(ptp["promised_date"])

print("PTP shape:", ptp.shape)
print("Unique PTP IDs:", ptp["ptp_id"].nunique())
print("\nPTP status distribution:")
print(ptp["status"].value_counts())

PTP shape: (18000, 15)
Unique PTP IDs: 18000

PTP status distribution:
status
BROKEN       4553
CANCELLED    4543
KEPT         4489
OPEN         4415
Name: count, dtype: int64


In [23]:
# Analyse PTP outcomes by source

ptp_source_analysis = pd.crosstab(
    ptp["source"],
    ptp["status"],
    margins=True
)

print(ptp_source_analysis)

# Calculate kept rate among resolved promises

resolved_ptp = ptp[
    ptp["status"].isin(["KEPT", "BROKEN"])
]

ptp_source_rates = (
    resolved_ptp.groupby("source")["status"]
    .apply(lambda x: (x == "KEPT").mean() * 100)
    .reset_index(name="kept_rate_pct")
)

print("\nKept rate by source:")
print(ptp_source_rates.round(2))

status    BROKEN  CANCELLED  KEPT  OPEN    All
source                                        
CALL        1169       1138  1176  1088   4571
FIELD       1152       1158  1114  1122   4546
SMS         1142       1097  1140  1070   4449
WHATSAPP    1090       1150  1059  1135   4434
All         4553       4543  4489  4415  18000

Kept rate by source:
     source  kept_rate_pct
0      CALL          50.15
1     FIELD          49.16
2       SMS          49.96
3  WHATSAPP          49.28


In [24]:
# Aggregate successful payments by account

successful_payment_summary = (
    successful_payments.groupby("account_id")
    .agg(
        successful_payment_count=("payment_id", "count"),
        total_successful_payment=("amount", "sum"),
        first_successful_payment=("event_at", "min"),
        last_successful_payment=("event_at", "max")
    )
    .reset_index()
)

# Link PTP records to successful payments

ptp_payment_analysis = ptp.merge(
    successful_payment_summary,
    on="account_id",
    how="left"
)

# Identify accounts with successful payments

ptp_payment_analysis["has_successful_payment"] = (
    ptp_payment_analysis["successful_payment_count"].notna()
)

print("Total PTP records:", len(ptp_payment_analysis))

print(
    "PTP records linked to accounts with successful payments:",
    ptp_payment_analysis["has_successful_payment"].sum()
)

print(
    "Percentage linked to successful payments:",
    round(
        ptp_payment_analysis["has_successful_payment"].mean() * 100,
        2
    ),
    "%"
)

Total PTP records: 18000
PTP records linked to accounts with successful payments: 8000
Percentage linked to successful payments: 44.44 %


In [25]:
# Check whether successful payments occurred after PTP creation

ptp_payment_analysis["payment_after_ptp"] = (
    ptp_payment_analysis["first_successful_payment"].notna()
    &
    (
        ptp_payment_analysis["first_successful_payment"]
        >= ptp_payment_analysis["event_at"]
    )
)

print(
    "PTP records with payment after promise creation:",
    ptp_payment_analysis["payment_after_ptp"].sum()
)

print(
    "Percentage of all PTP records:",
    round(
        ptp_payment_analysis["payment_after_ptp"].mean() * 100,
        2
    ),
    "%"
)

# Compare payment association by PTP status

print("\nPayment association by PTP status:")

print(
    ptp_payment_analysis.groupby("status")[
        "has_successful_payment"
    ].mean().mul(100).round(2)
)

PTP records with payment after promise creation: 3554
Percentage of all PTP records: 19.74 %

Payment association by PTP status:
status
BROKEN       43.88
CANCELLED    43.50
KEPT         46.18
OPEN         44.24
Name: has_successful_payment, dtype: float64


In [26]:
# Check whether payment occurred between PTP creation and promised date

ptp_payment_analysis["payment_within_promise_window"] = (
    ptp_payment_analysis["first_successful_payment"].notna()
    &
    (ptp_payment_analysis["first_successful_payment"] >=
     ptp_payment_analysis["event_at"])
    &
    (ptp_payment_analysis["first_successful_payment"] <=
     ptp_payment_analysis["promised_date"])
)

print(
    "Payments within promise window:",
    ptp_payment_analysis["payment_within_promise_window"].sum()
)

print(
    "Percentage of all PTP records:",
    round(
        ptp_payment_analysis["payment_within_promise_window"].mean() * 100,
        2
    ),
    "%"
)

print("\nPayment within promise window by PTP status:")

print(
    ptp_payment_analysis.groupby("status")[
        "payment_within_promise_window"
    ].mean().mul(100).round(2)
)

Payments within promise window: 487
Percentage of all PTP records: 2.71 %

Payment within promise window by PTP status:
status
BROKEN       2.53
CANCELLED    2.99
KEPT         2.83
OPEN         2.47
Name: payment_within_promise_window, dtype: float64


In [27]:
# Analyse repeated PTPs by account

ptp_account_frequency = (
    ptp.groupby("account_id")
    .agg(
        total_ptps=("ptp_id", "count"),
        unique_statuses=("status", "nunique"),
        first_ptp_date=("event_at", "min"),
        last_ptp_date=("event_at", "max")
    )
    .reset_index()
)

print("Unique accounts with PTPs:", len(ptp_account_frequency))

print(
    "Accounts with multiple PTPs:",
    (ptp_account_frequency["total_ptps"] > 1).sum()
)

print(
    "Maximum PTPs for one account:",
    ptp_account_frequency["total_ptps"].max()
)

print("\nDistribution of PTP frequency:")
print(ptp_account_frequency["total_ptps"].value_counts().sort_index())

Unique accounts with PTPs: 13532
Accounts with multiple PTPs: 3675
Maximum PTPs for one account: 6

Distribution of PTP frequency:
total_ptps
1    9857
2    2996
3     577
4      92
5       8
6       2
Name: count, dtype: int64


In [28]:
# Sort PTP records chronologically

ptp_sorted = ptp.sort_values(
    ["account_id", "event_at"]
).copy()

# Identify previous promise date for each account

ptp_sorted["previous_promised_date"] = (
    ptp_sorted.groupby("account_id")["promised_date"].shift(1)
)

ptp_sorted["previous_ptp_event"] = (
    ptp_sorted.groupby("account_id")["event_at"].shift(1)
)

# Check whether a new promise was created before the previous due date

ptp_sorted["new_ptp_before_previous_due"] = (
    ptp_sorted["previous_promised_date"].notna()
    &
    (
        ptp_sorted["event_at"] <
        ptp_sorted["previous_promised_date"]
    )
)

print(
    "Repeated PTP records:",
    ptp_sorted["previous_promised_date"].notna().sum()
)

print(
    "New promises created before previous due date:",
    ptp_sorted["new_ptp_before_previous_due"].sum()
)

print(
    "Percentage of repeated PTP records:",
    round(
        ptp_sorted.loc[
            ptp_sorted["previous_promised_date"].notna(),
            "new_ptp_before_previous_due"
        ].mean() * 100,
        2
    ),
    "%"
)

Repeated PTP records: 4468
New promises created before previous due date: 641
Percentage of repeated PTP records: 14.35 %


In [29]:
# Save PTP payment analysis

ptp_output_path = os.path.join(
    output_path,
    "ptp_payment_analysis.csv"
)

ptp_payment_analysis.to_csv(
    ptp_output_path,
    index=False
)

# Save PTP account frequency analysis

ptp_frequency_path = os.path.join(
    output_path,
    "ptp_account_frequency.csv"
)

ptp_account_frequency.to_csv(
    ptp_frequency_path,
    index=False
)

print("PTP analysis files saved successfully.")
print(ptp_output_path)
print(ptp_frequency_path)

PTP analysis files saved successfully.
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Outputs/ptp_payment_analysis.csv
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Outputs/ptp_account_frequency.csv


In [30]:
# Load cleaned field visits and complaints

field_visits = pd.read_csv(
    os.path.join(cleaned_path, "field_visits_cleaned.csv")
)

complaints = pd.read_csv(
    os.path.join(cleaned_path, "complaints_cleaned.csv")
)

# Convert event timestamps

field_visits["event_at"] = pd.to_datetime(
    field_visits["event_at"], errors="coerce"
)

complaints["event_at"] = pd.to_datetime(
    complaints["event_at"], errors="coerce"
)

print("Field visits shape:", field_visits.shape)
print("Complaints shape:", complaints.shape)

print("\nField visit outcomes:")
print(field_visits["outcome"].value_counts())

print("\nComplaint statuses:")
print(complaints["status"].value_counts())

Field visits shape: (25000, 11)
Complaints shape: (8000, 11)

Field visit outcomes:
outcome
WRONG_ADDRESS    4221
CONTACTED        4205
PTP              4200
NOT_AVAILABLE    4190
REFUSED          4104
PAID             4080
Name: count, dtype: int64

Complaint statuses:
status
OPEN             2096
INVESTIGATING    2005
CLOSED           1964
RESOLVED         1935
Name: count, dtype: int64


In [31]:
# Validate field visit and complaint account references

print("Field visit records:", len(field_visits))
print("Unique field visit IDs:", field_visits["visit_id"].nunique())

print("\nComplaint records:", len(complaints))
print("Unique complaint IDs:", complaints["complaint_id"].nunique())

# Check account references

print(
    "\nField visits with unmatched accounts:",
    (~field_visits["account_id"].isin(accounts["account_id"])).sum()
)

print(
    "Complaints with unmatched accounts:",
    (~complaints["account_id"].isin(accounts["account_id"])).sum()
)

# Check missing account IDs

print(
    "\nMissing account IDs in field visits:",
    field_visits["account_id"].isna().sum()
)

print(
    "Missing account IDs in complaints:",
    complaints["account_id"].isna().sum()
)

Field visit records: 25000
Unique field visit IDs: 25000

Complaint records: 8000
Unique complaint IDs: 8000

Field visits with unmatched accounts: 0
Complaints with unmatched accounts: 0

Missing account IDs in field visits: 0
Missing account IDs in complaints: 0


In [32]:
# Summarise field visit activity by account

field_visit_summary = (
    field_visits.groupby("account_id")
    .agg(
        total_visits=("visit_id", "count"),
        paid_visit_count=("outcome", lambda x: (x == "PAID").sum()),
        ptp_visit_count=("outcome", lambda x: (x == "PTP").sum()),
        wrong_address_count=("outcome", lambda x: (x == "WRONG_ADDRESS").sum())
    )
    .reset_index()
)

# Merge with actual successful recovery data

field_recovery_analysis = field_visit_summary.merge(
    account_recovery[["account_id", "total_recovered"]],
    on="account_id",
    how="left"
)

field_recovery_analysis["total_recovered"] = (
    field_recovery_analysis["total_recovered"].fillna(0)
)

field_recovery_analysis["has_successful_payment"] = (
    field_recovery_analysis["total_recovered"] > 0
)

# Compare accounts with and without a PAID field visit outcome

field_recovery_analysis["has_paid_visit"] = (
    field_recovery_analysis["paid_visit_count"] > 0
)

print(
    field_recovery_analysis.groupby("has_paid_visit").agg(
        accounts=("account_id", "count"),
        accounts_with_payment=("has_successful_payment", "sum"),
        average_recovery=("total_recovered", "mean")
    )
)

                accounts  accounts_with_payment  average_recovery
has_paid_visit                                                   
False              13098                   5828      43969.309016
True                3810                   1660      43889.755034


In [33]:
# Analyse recovery by field visit outcome

field_outcome_analysis = (
    field_visits.groupby("outcome")
    .agg(
        visit_records=("visit_id", "count"),
        unique_accounts=("account_id", "nunique")
    )
    .reset_index()
)

# Link account-level recovery

field_outcome_analysis = field_outcome_analysis.merge(
    field_visits[["account_id", "outcome"]]
    .drop_duplicates()
    .merge(
        account_recovery[["account_id", "total_recovered"]],
        on="account_id",
        how="left"
    )
    .groupby("outcome")
    .agg(
        accounts_with_payment=(
            "total_recovered",
            lambda x: (x.fillna(0) > 0).sum()
        ),
        average_recovery=("total_recovered", "mean")
    )
    .reset_index(),
    on="outcome",
    how="left"
)

field_outcome_analysis["payment_coverage_pct"] = (
    field_outcome_analysis["accounts_with_payment"]
    / field_outcome_analysis["unique_accounts"] * 100
)

print(field_outcome_analysis.round(2))

         outcome  visit_records  unique_accounts  accounts_with_payment  \
0      CONTACTED           4205             3927                   1733   
1  NOT_AVAILABLE           4190             3883                   1730   
2           PAID           4080             3810                   1660   
3            PTP           4200             3942                   1754   
4        REFUSED           4104             3849                   1673   
5  WRONG_ADDRESS           4221             3939                   1768   

   average_recovery  payment_coverage_pct  
0         101197.67                 44.13  
1          97844.38                 44.55  
2         100734.92                 43.57  
3          98790.49                 44.50  
4          98929.98                 43.47  
5          98597.18                 44.88  


In [34]:
# Aggregate complaint activity by account

complaint_summary = (
    complaints.groupby("account_id")
    .agg(
        total_complaints=("complaint_id", "count"),
        open_complaints=("status", lambda x: (x == "OPEN").sum()),
        investigating_complaints=(
            "status",
            lambda x: (x == "INVESTIGATING").sum()
        ),
        resolved_complaints=(
            "status",
            lambda x: (x == "RESOLVED").sum()
        ),
        closed_complaints=("status", lambda x: (x == "CLOSED").sum())
    )
    .reset_index()
)

# Connect complaints with account-level recovery

complaint_recovery_analysis = complaint_summary.merge(
    account_recovery[["account_id", "total_recovered"]],
    on="account_id",
    how="left"
)

complaint_recovery_analysis["total_recovered"] = (
    complaint_recovery_analysis["total_recovered"].fillna(0)
)

complaint_recovery_analysis["has_successful_payment"] = (
    complaint_recovery_analysis["total_recovered"] > 0
)

# Compare complaint-bearing accounts with recovery

print(
    complaint_recovery_analysis.agg(
        total_accounts=("account_id", "count"),
        accounts_with_payment=("has_successful_payment", "sum"),
        average_recovery=("total_recovered", "mean"),
        average_complaints=("total_complaints", "mean")
    )
)

                       account_id  has_successful_payment  total_recovered  \
total_accounts             7034.0                     NaN              NaN   
accounts_with_payment         NaN                  3066.0              NaN   
average_recovery              NaN                     NaN     43040.036834   
average_complaints            NaN                     NaN              NaN   

                       total_complaints  
total_accounts                      NaN  
accounts_with_payment               NaN  
average_recovery                    NaN  
average_complaints             1.137333  


In [35]:
# Categorise complaint accounts by complaint status

complaint_recovery_analysis["complaint_category"] = np.select(
    [
        complaint_recovery_analysis["open_complaints"] > 0,
        complaint_recovery_analysis["investigating_complaints"] > 0,
        complaint_recovery_analysis["resolved_complaints"] > 0,
        complaint_recovery_analysis["closed_complaints"] > 0
    ],
    [
        "OPEN",
        "INVESTIGATING",
        "RESOLVED",
        "CLOSED"
    ],
    default="OTHER"
)

# Summarise recovery by complaint category

complaint_status_analysis = (
    complaint_recovery_analysis.groupby("complaint_category")
    .agg(
        accounts=("account_id", "count"),
        accounts_with_payment=("has_successful_payment", "sum"),
        average_recovery=("total_recovered", "mean")
    )
    .reset_index()
)

complaint_status_analysis["payment_coverage_pct"] = (
    complaint_status_analysis["accounts_with_payment"]
    / complaint_status_analysis["accounts"] * 100
)

print(complaint_status_analysis.round(2))

  complaint_category  accounts  accounts_with_payment  average_recovery  \
0             CLOSED      1547                    671          42277.29   
1      INVESTIGATING      1815                    809          43658.71   
2               OPEN      2032                    889          42642.61   
3           RESOLVED      1640                    697          43567.26   

   payment_coverage_pct  
0                 43.37  
1                 44.57  
2                 43.75  
3                 42.50  


In [36]:
# Save field visit analysis

field_output_path = os.path.join(
    output_path,
    "field_visit_recovery_analysis.csv"
)

field_outcome_analysis.to_csv(
    field_output_path,
    index=False
)

# Save complaint analysis

complaint_output_path = os.path.join(
    output_path,
    "complaint_recovery_analysis.csv"
)

complaint_recovery_analysis.to_csv(
    complaint_output_path,
    index=False
)

complaint_status_path = os.path.join(
    output_path,
    "complaint_status_analysis.csv"
)

complaint_status_analysis.to_csv(
    complaint_status_path,
    index=False
)

print("Field visit and complaint analyses saved successfully.")

Field visit and complaint analyses saved successfully.


In [37]:
# Load cleaned account status history

account_status = pd.read_csv(
    os.path.join(cleaned_path, "account_status_history_cleaned.csv")
)

# Convert timestamps

account_status["event_at"] = pd.to_datetime(
    account_status["event_at"], errors="coerce"
)

account_status["recorded_at"] = pd.to_datetime(
    account_status["recorded_at"], errors="coerce"
)

print("Account status history shape:", account_status.shape)

print("\nUnique accounts:", account_status["account_id"].nunique())

print("\nStatus distribution:")
print(account_status["status"].value_counts())

Account status history shape: (60000, 15)

Unique accounts: 25999

Status distribution:
status
PAID          8650
CLOSED        8614
DELINQUENT    8612
NPA           8612
WRITEOFF      8583
ACTIVE        8518
PTP           8411
Name: count, dtype: int64


In [38]:
# Validate account references

print(
    "Unmatched account references:",
    (~account_status["account_id"].isin(accounts["account_id"])).sum()
)

print(
    "Missing account IDs:",
    account_status["account_id"].isna().sum()
)

# Examine status transitions

print("\nStatus transition distribution:")

print(
    account_status["status_transition"]
    .value_counts()
    .head(15)
)

# Examine unusual transitions

print(
    "\nUnusual transitions:",
    account_status["unusual_transition_flag"].sum()
)

# Examine recording-time inconsistencies

print(
    "Records recorded before event time:",
    account_status["recording_before_event_flag"].sum()
)

Unmatched account references: 0
Missing account IDs: 0

Status transition distribution:
status_transition
START -> CLOSED             3800
START -> NPA                3745
START -> ACTIVE             3733
START -> PAID               3725
START -> WRITEOFF           3718
START -> DELINQUENT         3717
START -> PTP                3561
NPA -> PAID                  735
CLOSED -> PAID               732
DELINQUENT -> CLOSED         729
WRITEOFF -> DELINQUENT       729
CLOSED -> DELINQUENT         725
DELINQUENT -> NPA            722
PTP -> PTP                   722
DELINQUENT -> DELINQUENT     720
Name: count, dtype: int64

Unusual transitions: 2168
Records recorded before event time: 30191


In [39]:
# Identify accounts with a PAID status event

paid_status_accounts = set(
    account_status.loc[
        account_status["status"] == "PAID",
        "account_id"
    ].unique()
)

# Identify accounts with successful payment transactions

successful_payment_accounts = set(
    successful_payments["account_id"].unique()
)

# Compare the two sets

both_paid = paid_status_accounts.intersection(
    successful_payment_accounts
)

status_only = paid_status_accounts - successful_payment_accounts

payment_only = successful_payment_accounts - paid_status_accounts

print("Accounts with PAID status:", len(paid_status_accounts))

print("Accounts with successful payments:", len(successful_payment_accounts))

print("Accounts appearing in both:", len(both_paid))

print("PAID status without successful payment:", len(status_only))

print("Successful payment without PAID status:", len(payment_only))

Accounts with PAID status: 7524
Accounts with successful payments: 13284
Accounts appearing in both: 3371
PAID status without successful payment: 4153
Successful payment without PAID status: 9913


In [40]:
# Save account status reconciliation

status_reconciliation = pd.DataFrame({
    "metric": [
        "Accounts with PAID status",
        "Accounts with successful payments",
        "Accounts appearing in both",
        "PAID status without successful payment",
        "Successful payment without PAID status"
    ],
    "account_count": [
        len(paid_status_accounts),
        len(successful_payment_accounts),
        len(both_paid),
        len(status_only),
        len(payment_only)
    ]
})

status_reconciliation.to_csv(
    os.path.join(output_path, "status_payment_reconciliation.csv"),
    index=False
)

print("Status reconciliation saved successfully.")

Status reconciliation saved successfully.


In [41]:
# Creating SQL workspace

sql_path = os.path.join(project_path, "SQL")
os.makedirs(sql_path, exist_ok=True)

database_path = os.path.join(
    project_path,
    "collections_analytics.db"
)

print("SQL folder:", sql_path)
print("Database path:", database_path)

SQL folder: /Users/simrannimje/Desktop/Collections_Analytics_Assignment/SQL
Database path: /Users/simrannimje/Desktop/Collections_Analytics_Assignment/collections_analytics.db


In [42]:
import sqlite3
import pandas as pd
import os

# Connect to SQLite database

conn = sqlite3.connect(database_path)

# Define datasets to load

datasets = {
    "accounts": "accounts_cleaned.csv",
    "payments": "payments_cleaned.csv",
    "ptp": "ptp_cleaned.csv",
    "field_visits": "field_visits_cleaned.csv",
    "complaints": "complaints_cleaned.csv",
    "account_status_history": "account_status_history_cleaned.csv",
    "calls": "calls_cleaned.csv",
    "call_dispositions": "call_dispositions_cleaned.csv",
    "agents": "agents_cleaned.csv",
    "agent_sessions": "agent_sessions_cleaned.csv",
    "whatsapp_events": "whatsapp_events_cleaned.csv",
    "sms_events": "sms_events_cleaned.csv"
}

# Load datasets into SQL tables

for table_name, file_name in datasets.items():

    file_path = os.path.join(cleaned_path, file_name)

    df = pd.read_csv(file_path)

    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    print(f"{table_name}: {df.shape[0]} rows loaded")

# Load golden account recovery dataset

golden = pd.read_csv(
    os.path.join(output_path, "golden_account_recovery.csv")
)

golden.to_sql(
    "golden_account_recovery",
    conn,
    if_exists="replace",
    index=False
)

print("\nGolden account recovery:", golden.shape[0], "rows")

print("\nAll datasets loaded successfully!")

accounts: 30000 rows loaded
payments: 25000 rows loaded
ptp: 18000 rows loaded
field_visits: 25000 rows loaded
complaints: 8000 rows loaded
account_status_history: 60000 rows loaded
calls: 90079 rows loaded
call_dispositions: 35000 rows loaded
agents: 30000 rows loaded
agent_sessions: 15000 rows loaded
whatsapp_events: 60000 rows loaded
sms_events: 45000 rows loaded

Golden account recovery: 30000 rows

All datasets loaded successfully!


In [43]:
# Verify SQLite tables and row counts

cursor = conn.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""")

tables = cursor.fetchall()

print("Tables available in database:\n")

for table in tables:
    table_name = table[0]

    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS total_rows FROM "{table_name}"',
        conn
    )["total_rows"].iloc[0]

    print(f"{table_name}: {count:,} rows")

Tables available in database:

account_status_history: 60,000 rows
accounts: 30,000 rows
agent_sessions: 15,000 rows
agents: 30,000 rows
call_dispositions: 35,000 rows
calls: 90,079 rows
complaints: 8,000 rows
field_visits: 25,000 rows
golden_account_recovery: 30,000 rows
payments: 25,000 rows
ptp: 18,000 rows
sms_events: 45,000 rows
whatsapp_events: 60,000 rows


In [44]:
# Check available cleaned datasets

cleaned_files = sorted(os.listdir(cleaned_path))

print("Available cleaned datasets:\n")

for file in cleaned_files:
    if file.endswith(".csv"):
        print(file)

print("\nTotal cleaned CSV files:", 
      len([f for f in cleaned_files if f.endswith(".csv")]))

Available cleaned datasets:

account_status_history_cleaned.csv
accounts_cleaned.csv
agent_sessions_cleaned.csv
agents_cleaned.csv
borrowers_cleaned.csv
call_attempts_cleaned.csv
call_dispositions_cleaned.csv
calls_cleaned.csv
campaigns_cleaned.csv
complaints_cleaned.csv
daily_targeting_cleaned.csv
field_visits_cleaned.csv
payments_cleaned.csv
ptp_cleaned.csv
sms_events_cleaned.csv
vendor_telephony_cleaned.csv
whatsapp_events_cleaned.csv

Total cleaned CSV files: 17


In [45]:
# Load the five remaining cleaned datasets into SQLite

remaining_datasets = {
    "borrowers": "borrowers_cleaned.csv",
    "campaigns": "campaigns_cleaned.csv",
    "daily_targeting": "daily_targeting_cleaned.csv",
    "call_attempts": "call_attempts_cleaned.csv",
    "vendor_telephony": "vendor_telephony_cleaned.csv"
}

for table_name, file_name in remaining_datasets.items():

    file_path = os.path.join(cleaned_path, file_name)

    df = pd.read_csv(file_path)

    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    print(f"{table_name}: {len(df):,} rows loaded")

print("\nAll five additional datasets loaded successfully!")

borrowers: 30,000 rows loaded
campaigns: 120 rows loaded
daily_targeting: 45,000 rows loaded
call_attempts: 120,000 rows loaded
vendor_telephony: 15 rows loaded

All five additional datasets loaded successfully!


In [46]:
# Final database inventory

cursor = conn.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""")

tables = [row[0] for row in cursor.fetchall()]

database_inventory = []

for table_name in tables:

    row_count = pd.read_sql_query(
        f'SELECT COUNT(*) AS total_rows FROM "{table_name}"',
        conn
    )["total_rows"].iloc[0]

    database_inventory.append({
        "table_name": table_name,
        "row_count": row_count
    })

database_inventory = pd.DataFrame(database_inventory)

print(database_inventory.to_string(index=False))

print("\nTotal tables:", len(database_inventory))
print("Total records:", database_inventory["row_count"].sum())

             table_name  row_count
 account_status_history      60000
               accounts      30000
         agent_sessions      15000
                 agents      30000
              borrowers      30000
          call_attempts     120000
      call_dispositions      35000
                  calls      90079
              campaigns        120
             complaints       8000
        daily_targeting      45000
           field_visits      25000
golden_account_recovery      30000
               payments      25000
                    ptp      18000
             sms_events      45000
       vendor_telephony         15
        whatsapp_events      60000

Total tables: 18
Total records: 666214


In [47]:
import pandas as pd
import os
import glob

project_path = "/Users/simrannimje/Desktop/Collections_Analytics_Assignment"
cleaned_path = os.path.join(project_path, "Data", "Cleaned")

csv_files = sorted(glob.glob(os.path.join(cleaned_path, "*.csv")))

print("Total cleaned CSV files:", len(csv_files))

for file in csv_files:
    df = pd.read_csv(file, nrows=5)
    print(f"\n{'='*70}")
    print("TABLE:", os.path.basename(file).replace("_cleaned.csv", ""))
    print("COLUMNS:", list(df.columns))
    print("DATA TYPES:")
    print(df.dtypes.to_string())

Total cleaned CSV files: 17

TABLE: account_status_history
COLUMNS: ['history_id', 'account_id', 'borrower_id', 'event_at', 'status', 'changed_by', 'source', 'recorded_at', 'recording_delay_hours', 'previous_status', 'status_transition', 'previous_event_at', 'hours_since_previous_status', 'unusual_transition_flag', 'recording_before_event_flag']
DATA TYPES:
history_id                         str
account_id                         str
borrower_id                        str
event_at                           str
status                             str
changed_by                         str
source                             str
recorded_at                        str
recording_delay_hours          float64
previous_status                    str
status_transition                  str
previous_event_at                  str
hours_since_previous_status    float64
unusual_transition_flag           bool
recording_before_event_flag       bool

TABLE: accounts
COLUMNS: ['account_id', 'borrower_id',

In [48]:
for file in csv_files:
    df = pd.read_csv(file)

    print(f"\n{'='*70}")
    print("TABLE:", os.path.basename(file))

    for col in df.columns:
        if df[col].dtype == "object":
            max_len = df[col].dropna().astype(str).str.len().max()
            print(f"{col}: max text length = {max_len}")

        elif pd.api.types.is_numeric_dtype(df[col]):
            print(
                f"{col}: min = {df[col].min()}, "
                f"max = {df[col].max()}"
            )


TABLE: account_status_history_cleaned.csv
recording_delay_hours: min = -23.99861111111111, max = 23.99944444444445
hours_since_previous_status: min = 0.0033333333333333, max = 5217.371944444444
unusual_transition_flag: min = False, max = True
recording_before_event_flag: min = False, max = True

TABLE: accounts_cleaned.csv
principal_amount: min = 10055.11, max = 799968.38
outstanding_amount: min = 1002.67, max = 699963.89
dpd: min = 0, max = 180
outstanding_above_principal_flag: min = False, max = True
borrower_id_missing_flag: min = False, max = True

TABLE: agent_sessions_cleaned.csv
session_duration_hours: min = 0.5, max = 9.998333333333331
valid_session_duration: min = True, max = True

TABLE: agents_cleaned.csv
timestamp_inconsistent: min = False, max = True

TABLE: borrowers_cleaned.csv
phone: min = 9000004704.0, max = 9999938435.0
phone_missing: min = False, max = True
email_missing: min = False, max = True
timestamp_inconsistent: min = False, max = True

TABLE: call_attempts_c

In [49]:
for file in csv_files:
    df = pd.read_csv(file)
    print(f"\n{'='*65}")
    print("TABLE:", os.path.basename(file))

    for col in df.columns:
        if (
            pd.api.types.is_string_dtype(df[col])
            or df[col].dtype == "object"
        ):
            values = df[col].dropna().astype(str)

            if not values.empty:
                print(
                    f"{col}: max length = {values.str.len().max()} "
                    f"| sample = {values.iloc[0]}"
                )


TABLE: account_status_history_cleaned.csv
history_id: max length = 14 | sample = HISTORY0037295
account_id: max length = 10 | sample = ACC0000001
borrower_id: max length = 10 | sample = BRW0009888
event_at: max length = 19 | sample = 2026-02-05 18:41:49
status: max length = 10 | sample = DELINQUENT
changed_by: max length = 10 | sample = AGT0000093
source: max length = 7 | sample = CALL
recorded_at: max length = 19 | sample = 2026-02-06 18:02:08
previous_status: max length = 10 | sample = PTP
status_transition: max length = 24 | sample = START -> DELINQUENT
previous_event_at: max length = 19 | sample = 2026-01-16 13:58:20

TABLE: accounts_cleaned.csv
account_id: max length = 10 | sample = ACC0000001
borrower_id: max length = 10 | sample = BRW0010742
loan_type: max length = 11 | sample = CONSUMER
risk_segment: max length = 6 | sample = MEDIUM
status: max length = 8 | sample = CLOSED
opened_at: max length = 19 | sample = 2025-11-11 04:37:00
timezone: max length = 12 | sample = Asia/Dubai